# 🤖 Q-learning: un robot que aprende a moverse por una rejilla 2D

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/samtzai/course_ai_for_industrial_control_students/blob/main/notebooks/07_reinforcement_learning/n15_grid_robot_qlearning_tutorial_training_process.ipynb)

> **Tiempo:** menos de 1 minuto en CPU. No necesita GPU.

Este notebook es una introducción sencilla al **Q-learning** con un mundo en rejilla: un robot tiene que llegar a la meta evitando trampas.
Es el primer paso antes de DQN (n16, Lunar Lander) y del carro-péndulo (n19), donde comparamos el RL con el control clásico.

## Descripción del problema

Un robot en una rejilla 2D tiene que:
- Salir de una posición fija
- Llegar a la celda de meta (+10 puntos)
- Evitar las celdas trampa (−10 puntos)
- Cada paso cuesta −1 punto (para premiar los caminos cortos)
- Encontrar el camino que maximiza la recompensa total

## La rejilla

- **Tamaño**: 5×5
- **Robot**: sale de la posición (0, 0), arriba a la izquierda
- **Meta**: la posición (4, 4) da +10
- **Trampas**: las posiciones (1, 1), (2, 3) y (3, 1) dan −10
- **Acciones**: arriba, abajo, izquierda, derecha
- **Coste por paso**: −1 por cada movimiento

Las posiciones son (fila, columna), con la fila 0 arriba, como en una matriz.

## Fundamentos del Q-learning

El Q-learning aprende la «calidad» (valor Q) de tomar cada acción en cada estado:

**Regla de actualización**: $Q(s, a) \leftarrow Q(s, a) + \alpha [r + \gamma \max_{a'} Q(s', a') - Q(s, a)]$

Donde:
- $s$: estado actual (posición)
- $a$: acción tomada
- $s'$: estado siguiente
- $r$: recompensa inmediata
- $\alpha$: tasa de aprendizaje (0.1)
- $\gamma$: factor de descuento (0.9)

🏭 **En la industria**, la misma idea aparece en la planificación de rutas de AGVs y robots móviles en almacenes, y en la secuenciación de tareas.

In [ ]:
IN_COLAB = 'google.colab' in str(get_ipython()) if 'get_ipython' in dir() else False
if IN_COLAB:
    from google.colab import output
    output.enable_custom_widget_manager()
print('Running in Colab' if IN_COLAB else 'Running locally')

In [ ]:
import os
import random
import warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.patches import Rectangle, Circle, FancyBboxPatch
from matplotlib.colors import ListedColormap
from IPython.display import HTML, display

warnings.filterwarnings('ignore', message='Animation was deleted')
try:
    import ipywidgets as widgets
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False
HEADLESS = os.environ.get('HEADLESS', '0') == '1'      # ejecución sin pantalla (p. ej. nbconvert): sin widgets
USE_WIDGETS = HAS_WIDGETS and not HEADLESS
ANIM_MODE = os.environ.get('ANIM_MODE', 'jshtml')      # 'jshtml' (reproductor), 'static' (fotogramas) o 'none'
SAVE_GIFS = True
FIG_DIR = Path('./figures_qlearning'); FIG_DIR.mkdir(exist_ok=True)

# Estilo visual común con n19
COL = dict(ink='#0f172a', muted='#64748b', grid='#cbd5e1', robot='#ea580c', goal='#16a34a', trap='#dc2626', start='#2563eb', path='#ea580c')
plt.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': 'white', 'axes.edgecolor': '#cbd5e1',
                     'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.color': '#eef2f7',
                     'font.size': 9, 'axes.titlesize': 10, 'axes.titleweight': 'bold', 'animation.embed_limit': 60, 'figure.dpi': 80})

# Semilla para que los resultados sean reproducibles
np.random.seed(42)
random.seed(42)
ACTION_NAMES = ['Arriba', 'Abajo', 'Izquierda', 'Derecha']
print(f'widgets: {USE_WIDGETS} | animaciones: {ANIM_MODE}')

## El entorno

In [ ]:
class GridWorld:
    def __init__(self, size=5, step_reward=-1, goal_reward=10, trap_reward=-10):
        self.size = size
        self.start_pos = (0, 0)
        self.goal_pos = (4, 4)
        self.traps = [(1, 1), (2, 3), (3, 1)]  # posiciones de las trampas

        # Recompensas
        self.goal_reward = goal_reward
        self.trap_reward = trap_reward
        self.step_reward = step_reward

        # Acciones: 0=arriba, 1=abajo, 2=izquierda, 3=derecha (cambio de fila, cambio de columna)
        self.actions = ACTION_NAMES
        self.action_effects = [(-1, 0), (1, 0), (0, -1), (0, 1)]

        self.reset()

    def reset(self):
        """Devuelve el robot a la posición de salida"""
        self.robot_pos = list(self.start_pos)
        return tuple(self.robot_pos)

    def is_valid_position(self, pos):
        """¿Está la posición dentro de la rejilla?"""
        row, col = pos
        return 0 <= row < self.size and 0 <= col < self.size

    def get_reward(self, pos):
        """Recompensa por estar en una posición"""
        if pos == self.goal_pos:
            return self.goal_reward
        elif pos in self.traps:
            return self.trap_reward
        else:
            return self.step_reward

    def step(self, action):
        """Ejecuta una acción y devuelve (estado_siguiente, recompensa, fin)"""
        dr, dc = self.action_effects[action]
        new_pos = (self.robot_pos[0] + dr, self.robot_pos[1] + dc)

        # Si la nueva posición se sale de la rejilla, el robot se queda donde está
        if self.is_valid_position(new_pos):
            self.robot_pos = list(new_pos)

        current_pos = tuple(self.robot_pos)
        reward = self.get_reward(current_pos)

        # El episodio termina al llegar a la meta o caer en una trampa
        done = (current_pos == self.goal_pos) or (current_pos in self.traps)

        return current_pos, reward, done

    def visualize(self, q_table=None, show_values=False, ax=None):
        """Dibuja la rejilla (y, opcionalmente, el valor máximo de Q en cada celda)"""
        own = ax is None
        if own:
            fig, ax = plt.subplots(figsize=(4.5, 4.5))
        draw_grid(ax, self)
        for i in range(self.size):
            for j in range(self.size):
                if show_values and q_table is not None and (i, j) not in self.traps and (i, j) != self.goal_pos:
                    ax.text(j, i + 0.32, f'{np.max(q_table[i, j]):.1f}', ha='center', va='center', fontsize=8, color=COL['muted'])
        ax.add_patch(Circle((self.robot_pos[1], self.robot_pos[0]), 0.28, color=COL['robot'], zorder=5))
        ax.set_title('Mundo en rejilla: navegación del robot')
        if own:
            plt.tight_layout(); plt.show()


def draw_grid(ax, env):
    """Fondo común: celdas, salida, meta y trampas (fila 0 arriba)."""
    ax.set_xlim(-0.5, env.size - 0.5); ax.set_ylim(env.size - 0.5, -0.5)
    ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    for sp in ax.spines.values():
        sp.set_visible(False)
    for i in range(env.size):
        for j in range(env.size):
            fc = 'white'
            if (i, j) == env.goal_pos:
                fc = '#dcfce7'
            elif (i, j) in env.traps:
                fc = '#fee2e2'
            elif (i, j) == env.start_pos:
                fc = '#dbeafe'
            ax.add_patch(FancyBboxPatch((j - 0.46, i - 0.46), 0.92, 0.92, boxstyle='round,pad=0,rounding_size=0.08',
                                        fc=fc, ec=COL['grid'], lw=1, zorder=0))
            if (i, j) == env.goal_pos:
                ax.text(j, i - 0.25, f'META\n+{env.goal_reward}', ha='center', va='center', fontsize=7, weight='bold', color=COL['goal'])
            elif (i, j) in env.traps:
                ax.text(j, i - 0.25, f'TRAMPA\n{env.trap_reward}', ha='center', va='center', fontsize=7, weight='bold', color=COL['trap'])
            elif (i, j) == env.start_pos:
                ax.text(j, i - 0.32, 'SALIDA', ha='center', va='center', fontsize=7, weight='bold', color=COL['start'])


# Crear y dibujar el entorno
env = GridWorld()
print('Entorno en rejilla:')
print(f'- Tamaño: {env.size}x{env.size}')
print(f'- Salida: {env.start_pos}')
print(f'- Meta: {env.goal_pos} (recompensa: +{env.goal_reward})')
print(f'- Trampas: {env.traps} (recompensa: {env.trap_reward} cada una)')
print(f'- Coste por paso: {env.step_reward}')
print(f'- Acciones: {env.actions}')

env.visualize()

## El agente de Q-learning

In [ ]:
class QLearningAgent:
    def __init__(self, grid_size, n_actions, learning_rate=0.1, discount_factor=0.9, epsilon=1.0):
        self.grid_size = grid_size
        self.n_actions = n_actions
        self.learning_rate = learning_rate
        self.discount_factor = discount_factor
        self.epsilon = epsilon

        # Tabla Q inicializada a ceros. Forma: (filas, columnas, acciones)
        self.q_table = np.zeros((grid_size, grid_size, n_actions))

        # Seguimiento para visualizar y analizar
        self.q_history = []        # copias de la tabla Q
        self.epsilon_history = []  # valores de ε
        self.q_updates = []        # ejemplos de actualizaciones de Q
        self.action_counts = {'explore': 0, 'exploit': 0}  # exploración frente a explotación

    def choose_action(self, state, track_policy=False):
        """Elige una acción con la política ε-greedy"""
        explore = random.random() < self.epsilon

        if explore:
            # Explorar: acción aleatoria
            action = random.randint(0, self.n_actions - 1)
            if track_policy:
                self.action_counts['explore'] += 1
        else:
            # Explotar: la mejor acción según la tabla Q
            row, col = state
            action = int(np.argmax(self.q_table[row, col]))
            if track_policy:
                self.action_counts['exploit'] += 1

        if track_policy:
            return action, explore
        else:
            return action

    def update_q_table(self, state, action, reward, next_state, done, track_update=False):
        """Actualiza la tabla Q con la regla del Q-learning"""
        row, col = state
        next_row, next_col = next_state

        # Valor Q actual
        current_q = self.q_table[row, col, action]

        # Máximo valor Q del estado siguiente (0 si es terminal)
        if done:
            max_next_q = 0
        else:
            max_next_q = np.max(self.q_table[next_row, next_col])

        # Regla del Q-learning
        td_target = reward + self.discount_factor * max_next_q
        td_error = td_target - current_q
        new_q = current_q + self.learning_rate * td_error

        # Guardar la información de la actualización para analizarla
        if track_update:
            self.q_updates.append({
                'state': state, 'action': action, 'reward': reward, 'next_state': next_state,
                'current_q': current_q, 'max_next_q': max_next_q, 'td_target': td_target,
                'td_error': td_error, 'new_q': new_q, 'done': done})

        self.q_table[row, col, action] = new_q

    def decay_epsilon(self, decay_rate=0.995, min_epsilon=0.01):
        """Reduce la tasa de exploración"""
        self.epsilon = max(self.epsilon * decay_rate, min_epsilon)
        self.epsilon_history.append(self.epsilon)

    def save_q_snapshot(self, episode):
        """Guarda una copia de la tabla Q actual"""
        self.q_history.append({'episode': episode, 'q_table': self.q_table.copy(), 'epsilon': self.epsilon})

    def get_policy(self):
        """Política aprendida: la mejor acción en cada estado"""
        return np.argmax(self.q_table, axis=2)


# Crear el agente
agent = QLearningAgent(grid_size=5, n_actions=4, learning_rate=0.1, discount_factor=0.9, epsilon=1.0)

print('Agente de Q-learning creado:')
print(f'- Rejilla: {agent.grid_size}x{agent.grid_size}')
print(f'- Número de acciones: {agent.n_actions}')
print(f'- Tasa de aprendizaje (α): {agent.learning_rate}')
print(f'- Factor de descuento (γ): {agent.discount_factor}')
print(f'- Exploración inicial (ε): {agent.epsilon}')
print(f'- Forma de la tabla Q: {agent.q_table.shape}')
print('- Con seguimiento de la evolución de Q, de ε y de las actualizaciones')

## Entrenamiento del agente

## La política ε-greedy

Antes de entrenar, veamos cómo elige las acciones la política ε-greedy.

In [ ]:
def demonstrate_epsilon_greedy_policy(agent, state=(2, 2), n_samples=1000):
    """Muestra la política ε-greedy en acción"""
    epsilon_values = [1.0, 0.5, 0.1, 0.0]
    fig, axes = plt.subplots(2, 2, figsize=(7, 5.5))
    axes = axes.flatten()

    for idx, epsilon_test in enumerate(epsilon_values):
        original_epsilon = agent.epsilon
        agent.epsilon = epsilon_test
        action_counts = {i: 0 for i in range(4)}
        for _ in range(n_samples):
            action_counts[agent.choose_action(state)] += 1
        agent.epsilon = original_epsilon

        ax = axes[idx]
        percentages = [action_counts[i] / n_samples * 100 for i in range(4)]
        bars = ax.bar(ACTION_NAMES, percentages, color=['#93c5fd', '#fca5a5', '#86efac', '#fcd34d'])
        for bar, pct in zip(bars, percentages):
            ax.text(bar.get_x() + bar.get_width() / 2., bar.get_height() + 1, f'{pct:.1f}%', ha='center', va='bottom', fontsize=8)
        label = 'solo exploración' if epsilon_test == 1.0 else 'solo explotación' if epsilon_test == 0.0 else f'{epsilon_test * 100:.0f}% exploración'
        ax.set_title(f'ε = {epsilon_test} ({label})')
        ax.set_ylabel('Frecuencia (%)'); ax.set_ylim(0, 110)
        q_vals = agent.q_table[state]
        ax.text(0.02, 0.98, f'Q: {[f"{q:.2f}" for q in q_vals]}\nMejor acción: {ACTION_NAMES[int(np.argmax(q_vals))]}',
                transform=ax.transAxes, va='top', fontsize=7, bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

    plt.suptitle(f'Política ε-greedy · estado {state} · {n_samples} muestras', weight='bold')
    plt.tight_layout(); plt.show()

    print('Cómo funciona ε-greedy:')
    print('• ε = 1.0: solo exploración, todas las acciones al azar (25 % cada una)')
    print('• ε = 0.5: 50 % exploración, 50 % explotación')
    print('• ε = 0.1: 10 % exploración, 90 % explotación')
    print('• ε = 0.0: solo explotación, siempre la mejor acción conocida')
    print('\nEste equilibrio permite al agente:')
    print('1. Probar acciones nuevas al principio (ε alto)')
    print('2. Pasar poco a poco a aprovechar lo aprendido (ε decreciente)')
    print('3. Mantener algo de exploración por si el entorno cambia')
    print('\n(Con la tabla Q aún a ceros, la "mejor" acción es la primera: Arriba.)')


demonstrate_epsilon_greedy_policy(agent)

In [ ]:
# Parámetros del entrenamiento
n_episodes = 1000
max_steps_per_episode = 50

# Seguimiento del progreso
episode_rewards = []
episode_lengths = []
success_rate = []

# Episodios en los que guardamos una copia de la tabla Q
snapshot_episodes = [0, 10, 50, 100, 200, 500, 999]

# Episodios que grabamos paso a paso para la animación (robot + tabla Q)
ANIM_EPISODES = [0, 1, 5, 20, 60, 150, 400, 999]
anim_frames = []

print('Empieza el entrenamiento con Q-learning...')
print(f'Episodios: {n_episodes}, pasos máximos por episodio: {max_steps_per_episode}')
print(f'Copias de Q en los episodios: {snapshot_episodes}')
print()

for episode in range(n_episodes):
    state = env.reset()
    total_reward = 0
    steps = 0
    agent.action_counts = {'explore': 0, 'exploit': 0}
    record = episode in ANIM_EPISODES
    if record:
        anim_frames.append(dict(ep=episode, pos=state, q=agent.q_table.copy(), eps=agent.epsilon, ret=0, a=None, explored=False))

    for step in range(max_steps_per_episode):
        # Elegir acción (registramos exploración/explotación en todos los episodios)
        action, explored = agent.choose_action(state, track_policy=True)

        # Ejecutar la acción
        next_state, reward, done = env.step(action)

        # Actualizar la tabla Q (guardamos ejemplos de las primeras actualizaciones)
        track_update = episode < 3 and len(agent.q_updates) < 20
        agent.update_q_table(state, action, reward, next_state, done, track_update=track_update)

        state = next_state
        total_reward += reward
        steps += 1
        if record:
            anim_frames.append(dict(ep=episode, pos=state, q=agent.q_table.copy(), eps=agent.epsilon, ret=total_reward, a=action, explored=explored))

        if done:
            break

    # Reducir la exploración
    agent.decay_epsilon()

    # Guardar copia de la tabla Q en algunos episodios
    if episode in snapshot_episodes:
        agent.save_q_snapshot(episode)

    episode_rewards.append(total_reward)
    episode_lengths.append(steps)

    # Tasa de éxito (llegar a la meta) en los últimos 100 episodios
    if episode >= 99:
        recent_rewards = episode_rewards[-100:]
        successes = sum(1 for r in recent_rewards if r > 0)  # recompensa positiva = ha llegado a la meta
        success_rate.append(successes / 100.0)

    if episode < 5:
        n_act = agent.action_counts['explore'] + agent.action_counts['exploit']
        explore_pct = agent.action_counts['explore'] / n_act * 100
        print(f'Episodio {episode + 1:2d}: recompensa = {total_reward:6.1f}, pasos = {steps:2d}, '
              f'ε = {agent.epsilon:.3f}, exploración = {explore_pct:.1f}% ({agent.action_counts["explore"]}/{n_act} acciones)')
    elif (episode + 1) % 100 == 0:
        print(f'Episodio {episode + 1:4d}: recompensa media = {np.mean(episode_rewards[-100:]):6.2f}, '
              f'duración media = {np.mean(episode_lengths[-100:]):5.1f}, tasa de éxito = {success_rate[-1]:.0%}, ε = {agent.epsilon:.3f}')

print('\n¡Entrenamiento terminado!')
print(f'Copias de la tabla Q guardadas: {len(agent.q_history)}')
print(f'Ejemplos de actualización guardados: {len(agent.q_updates)}')
print(f'Fotogramas grabados para la animación: {len(anim_frames)}')

## 🎬 El entrenamiento en movimiento

A la izquierda, el robot en algunos episodios del entrenamiento (borde naranja continuo: acción elegida explotando lo aprendido;
discontinuo: acción al azar por exploración). A la derecha, la tabla Q **en ese mismo instante**: el color es $\max_a Q(s,a)$ y la flecha,
la mejor acción en cada celda. Fíjate en cómo los valores se propagan **hacia atrás** desde la meta, episodio a episodio.

In [ ]:
ARROW = {0: (0, -0.3), 1: (0, 0.3), 2: (-0.3, 0), 3: (0.3, 0)}   # (dx, dy) en el dibujo; la fila 0 está arriba


def draw_q_panel(ax, env, q, vmin=-10, vmax=10):
    """Mapa de max Q con flechas de la política greedy. Devuelve los objetos para actualizarlos."""
    ax.set_xlim(-0.5, env.size - 0.5); ax.set_ylim(env.size - 0.5, -0.5)
    ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    for sp in ax.spines.values():
        sp.set_visible(False)
    im = ax.imshow(q.max(2), cmap='RdYlGn', vmin=vmin, vmax=vmax, zorder=0)
    texts, arrows = {}, {}
    for i in range(env.size):
        for j in range(env.size):
            if (i, j) in env.traps or (i, j) == env.goal_pos:
                ax.text(j, i, 'META' if (i, j) == env.goal_pos else 'TRAMPA', ha='center', va='center', fontsize=7, weight='bold', color=COL['ink'])
                continue
            texts[i, j] = ax.text(j, i + 0.3, '', ha='center', va='center', fontsize=7, color=COL['ink'])
            arrows[i, j] = ax.annotate('', xy=(j, i), xytext=(j, i), arrowprops=dict(arrowstyle='-|>', color=COL['ink'], lw=1.5))
    for k in range(env.size + 1):
        ax.axhline(k - 0.5, color='white', lw=2); ax.axvline(k - 0.5, color='white', lw=2)
    return im, texts, arrows


def update_q_panel(panel, q):
    im, texts, arrows = panel
    im.set_data(q.max(2))
    for (i, j), t in texts.items():
        t.set_text(f'{q[i, j].max():.1f}')
        a = arrows[i, j]
        if np.allclose(q[i, j], q[i, j, 0]):      # todas iguales (sin información): sin flecha
            a.xy = a.xyann = (j, i - 0.05)
        else:
            dx, dy = ARROW[int(np.argmax(q[i, j]))]
            a.xyann = (j - dx * 0.6, i - 0.05 - dy * 0.6); a.xy = (j + dx, i - 0.05 + dy)


def animate_training(frames, env, every=1, fps=6):
    fig, (axg, axq) = plt.subplots(1, 2, figsize=(9, 4.4))
    draw_grid(axg, env)
    robot = Circle(frames[0]['pos'][::-1], 0.25, color=COL['robot'], zorder=5); axg.add_patch(robot)
    ring = Circle(frames[0]['pos'][::-1], 0.34, fill=False, ec=COL['robot'], lw=2, zorder=5); axg.add_patch(ring)
    trail, = axg.plot([], [], color=COL['path'], lw=2, alpha=0.35, zorder=4)
    info = axg.set_title('')
    panel = draw_q_panel(axq, env, frames[0]['q'])
    axq.set_title('Tabla Q: max Q(s,a) y mejor acción')
    plt.tight_layout()
    state = dict(ep=None, path=[])

    def upd(k):
        f = frames[k]
        if f['ep'] != state['ep']:
            state['ep'], state['path'] = f['ep'], []
        state['path'].append(f['pos'])
        p = np.array(state['path'])
        trail.set_data(p[:, 1], p[:, 0])
        robot.center = ring.center = (f['pos'][1], f['pos'][0])
        ring.set_linestyle('--' if f['explored'] else '-')
        act = '' if f['a'] is None else f' · {ACTION_NAMES[f["a"]]}{" (azar)" if f["explored"] else ""}'
        info.set_text(f'Episodio {f["ep"] + 1} · ε = {f["eps"]:.2f} · recompensa {f["ret"]:+.0f}{act}')
        update_q_panel(panel, f['q'])
        return []

    anim = animation.FuncAnimation(fig, upd, frames=range(0, len(frames), every), interval=1000 / fps)
    anim.upd, anim.frame_list, anim.fig = upd, list(range(0, len(frames), every)), fig
    plt.close(fig)
    return anim


def filmstrip(anim, n=4):
    """Alternativa sin JavaScript: varios fotogramas en una sola figura."""
    ks = [anim.frame_list[int(i)] for i in np.linspace(0, len(anim.frame_list) - 1, n)]
    fig, axs = plt.subplots(n, 1, figsize=(8, 3.9 * n))
    for ax, k in zip(np.atleast_1d(axs), ks):
        anim.upd(k); anim.fig.canvas.draw(); ax.imshow(np.asarray(anim.fig.canvas.buffer_rgba())); ax.axis('off')
    plt.tight_layout(); plt.show()


def show_anim(anim, gif=None, fps=6, as_gif=False):
    if SAVE_GIFS and gif:
        anim.save(FIG_DIR / gif, writer=animation.PillowWriter(fps=fps), dpi=72)
        print(f'GIF guardado en {FIG_DIR / gif}')
    if as_gif and ANIM_MODE != 'none':
        tmp = FIG_DIR / '_tmp.gif'; anim.save(tmp, writer=animation.PillowWriter(fps=fps), dpi=72)
        from IPython.display import Image
        display(Image(data=tmp.read_bytes(), format='gif'))
    elif ANIM_MODE == 'jshtml':
        display(HTML(anim.to_jshtml(default_mode='once')))
    elif ANIM_MODE == 'static':
        filmstrip(anim)


show_anim(animate_training(anim_frames, env), gif='n15_entrenamiento_qlearning.gif')

## Visualizar el progreso del entrenamiento

## Ejemplos de actualizaciones de Q durante el entrenamiento

Veamos con detalle algunas actualizaciones de Q de los primeros episodios para entender cómo aprende el agente.

In [ ]:
def display_q_updates(agent, max_updates=10):
    """Muestra con detalle ejemplos de actualizaciones de Q"""
    if not agent.q_updates:
        print('No se guardó ninguna actualización de Q.')
        return

    print('🔍 Ejemplos de actualización del Q-learning')
    print('=' * 70)
    print("Regla: Q(s,a) ← Q(s,a) + α[r + γ⋅max Q(s',a') - Q(s,a)]")
    print(f'Parámetros: α = {agent.learning_rate}, γ = {agent.discount_factor}')
    print('=' * 70)

    for i, update in enumerate(agent.q_updates[:max_updates]):
        print(f"\n📍 Actualización #{i + 1}:")
        print(f"   Estado: {update['state']} → Acción: {ACTION_NAMES[update['action']]} → Estado siguiente: {update['next_state']}")
        print(f"   Recompensa: {update['reward']}")
        print(f"   ¿Terminal?: {'Sí' if update['done'] else 'No'}")
        print('\n   🧮 Cálculo:')
        print(f"   • Q(s,a) actual = {update['current_q']:.3f}")
        print(f"   • Recompensa (r) = {update['reward']}")
        if update['done']:
            print("   • max Q(s',a') = 0 (estado terminal)")
        else:
            print(f"   • max Q(s',a') = {update['max_next_q']:.3f}")
        print(f"   • Objetivo TD = r + γ⋅max Q(s',a') = {update['reward']} + {agent.discount_factor}×{update['max_next_q']:.3f} = {update['td_target']:.3f}")
        print(f"   • Error TD = {update['td_target']:.3f} - {update['current_q']:.3f} = {update['td_error']:.3f}")
        print(f"   • Nuevo Q(s,a) = {update['current_q']:.3f} + {agent.learning_rate}×{update['td_error']:.3f} = {update['new_q']:.3f}")
        change = update['new_q'] - update['current_q']
        direction = '↗️' if change > 0 else '↘️' if change < 0 else '➡️'
        print(f'   • Cambio: {change:+.3f} {direction}')
        print('-' * 50)


display_q_updates(agent, max_updates=8)

## Evolución de la función Q durante el entrenamiento

Cómo pasan los valores Q de estar todos a cero a los valores aprendidos.

In [ ]:
def visualize_q_evolution(agent, env):
    """Evolución de los valores Q durante el entrenamiento"""
    if not agent.q_history:
        print('No se guardó ninguna copia de la tabla Q.')
        return
    n = min(len(agent.q_history), 8)
    fig, axes = plt.subplots(2, 4, figsize=(15, 7.5))
    for idx, snapshot in enumerate(agent.q_history[:8]):
        ax = axes[idx // 4, idx % 4]
        panel = draw_q_panel(ax, env, snapshot['q_table'])
        update_q_panel(panel, snapshot['q_table'])
        ax.set_title(f'Episodio {snapshot["episode"]} · ε = {snapshot["epsilon"]:.3f}')
    for idx in range(n, 8):
        axes[idx // 4, idx % 4].set_visible(False)
    plt.suptitle('Evolución de la función Q: máximo valor Q por estado y mejor acción', weight='bold')
    plt.tight_layout()
    fig.colorbar(plt.cm.ScalarMappable(norm=plt.Normalize(-10, 10), cmap='RdYlGn'), ax=axes, shrink=0.7, label='max Q')
    plt.show()

    plt.figure(figsize=(7, 3))
    plt.plot(agent.epsilon_history, lw=2, color='#7c3aed')
    plt.title('Decaimiento de ε durante el entrenamiento'); plt.xlabel('Episodio'); plt.ylabel('ε')
    plt.axhline(y=0.01, color=COL['trap'], ls='--', alpha=0.7, label='ε mínimo (0.01)'); plt.legend()
    plt.show()

    print('🎯 Qué se ve en la evolución de Q:')
    print('• Al principio (episodio 0) todos los valores son cero')
    print('• Los valores se aprenden poco a poco con la experiencia')
    print('• Las celdas cerca de la meta tienen valores positivos altos (verde)')
    print('• Junto a las trampas los valores son negativos (rojo)')
    print(f"• ε bajó de {agent.q_history[0]['epsilon']:.3f} a {agent.q_history[-1]['epsilon']:.3f}")
    print('• Los valores se estabilizan cuando baja la exploración')


visualize_q_evolution(agent, env)

In [ ]:
def visualize_action_q_evolution(agent, state=(2, 2)):
    """Cómo evolucionan los valores Q de cada acción en un estado concreto"""
    if not agent.q_history:
        print('No se guardó ninguna copia de la tabla Q.')
        return
    colors = ['#3b82f6', '#ef4444', '#22c55e', '#f59e0b']
    episodes = [s['episode'] for s in agent.q_history]
    q_evolution = {a: [s['q_table'][state][a] for s in agent.q_history] for a in range(4)}

    plt.figure(figsize=(7, 4))
    for a in range(4):
        plt.plot(episodes, q_evolution[a], marker='o', lw=2, color=colors[a], label=ACTION_NAMES[a], ms=5)
    plt.title(f'Evolución de Q en el estado {state}'); plt.xlabel('Episodio'); plt.ylabel('Valor Q')
    plt.legend(title='Acciones'); plt.axhline(y=0, color='black', alpha=0.3)
    final = [q_evolution[a][-1] for a in range(4)]
    plt.text(0.02, 0.98, f'Mejor acción final: {ACTION_NAMES[int(np.argmax(final))]}\nQ finales: {[f"{q:.2f}" for q in final]}',
             transform=plt.gca().transAxes, va='top', fontsize=8, bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    plt.tight_layout(); plt.show()

    print(f'\n📊 Resumen de la evolución de Q en el estado {state}')
    print('=' * 72)
    print(f"{'Episodio':<10} {'Arriba':<9} {'Abajo':<9} {'Izq.':<9} {'Der.':<9} {'Mejor acción'}")
    print('-' * 72)
    for i, ep in enumerate(episodes):
        q = [q_evolution[a][i] for a in range(4)]
        print(f'{ep:<10} {q[0]:<9.2f} {q[1]:<9.2f} {q[2]:<9.2f} {q[3]:<9.2f} {ACTION_NAMES[int(np.argmax(q))]}')


visualize_action_q_evolution(agent, state=(2, 2))

In [ ]:
def moving_average(x, w=50):
    x = np.asarray(x, float)
    return np.convolve(x, np.ones(w) / w, mode='valid') if len(x) >= w else x


def plot_training(episode_rewards, episode_lengths, success_rate, epsilon_history, q_table, title=''):
    fig, axs = plt.subplots(1, 4, figsize=(16, 3.4))
    axs[0].plot(episode_rewards, '.', ms=2, alpha=0.3, color='#2563eb', label='episodio')
    axs[0].plot(np.arange(len(moving_average(episode_rewards))) + 49, moving_average(episode_rewards), color='#2563eb', lw=2, label='media móvil (50)')
    axs[0].set(title='Recompensa por episodio', xlabel='Episodio'); axs[0].legend(fontsize=7, frameon=False)
    axs[1].plot(episode_lengths, '.', ms=2, alpha=0.3, color='#16a34a')
    axs[1].plot(np.arange(len(moving_average(episode_lengths))) + 49, moving_average(episode_lengths), color='#16a34a', lw=2)
    axs[1].set(title='Duración del episodio (pasos)', xlabel='Episodio')
    axs[2].plot(range(100, 100 + len(success_rate)), success_rate, color='#7c3aed', lw=2, label='tasa de éxito (100 ep.)')
    axs[2].plot(epsilon_history, color='#f59e0b', lw=2, label='ε')
    axs[2].set(title='Tasa de éxito y ε', xlabel='Episodio', ylim=(0, 1.05)); axs[2].legend(fontsize=7, frameon=False)
    panel = draw_q_panel(axs[3], GridWorld(), q_table); update_q_panel(panel, q_table); axs[3].set_title('Política final')
    if title:
        fig.suptitle(title, x=0.01, ha='left', weight='bold')
    plt.tight_layout(); plt.show()


plot_training(episode_rewards, episode_lengths, success_rate, agent.epsilon_history, agent.q_table)

print('Estadísticas del entrenamiento:')
print(f'- Episodios: {len(episode_rewards)}')
print(f'- Recompensa media (últimos 100 episodios): {np.mean(episode_rewards[-100:]):.2f}')
print(f'- Duración media (últimos 100 episodios): {np.mean(episode_lengths[-100:]):.1f} pasos')
if success_rate:
    print(f'- Tasa de éxito final: {success_rate[-1]:.0%}')
print(f'- Exploración final (ε): {agent.epsilon:.3f}')
print(f'- Decaimiento de ε: {agent.epsilon_history[0]:.3f} → {agent.epsilon_history[-1]:.3f}')
print(f'- Actualizaciones de Q guardadas: {len(agent.q_updates)}')
print(f'- Copias de la tabla Q: {len(agent.q_history)}')

## La política aprendida

In [ ]:
def visualize_policy(agent, env):
    """Política aprendida (flechas) y máximo valor Q por estado"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4.8))
    policy = agent.get_policy()
    draw_grid(ax1, env)
    for i in range(env.size):
        for j in range(env.size):
            if (i, j) in env.traps or (i, j) == env.goal_pos:
                continue
            dx, dy = ARROW[policy[i, j]]
            ax1.annotate('', xy=(j + dx, i + 0.1 + dy), xytext=(j - dx * 0.6, i + 0.1 - dy * 0.6),
                         arrowprops=dict(arrowstyle='-|>', color=COL['ink'], lw=2))
    ax1.set_title('Política aprendida (mejor acción)')
    panel = draw_q_panel(ax2, env, agent.q_table); update_q_panel(panel, agent.q_table)
    ax2.set_title('Máximo valor Q por estado')
    plt.colorbar(panel[0], ax=ax2, label='max Q')
    plt.tight_layout(); plt.savefig(FIG_DIR / 'n15_politica.png', dpi=110, bbox_inches='tight'); plt.show()


visualize_policy(agent, env)
print('Cómo leer la política:')
print('- Las flechas indican la mejor acción desde cada estado')
print('- Valores Q altos (verde): estados más valiosos')
print('- Valores Q bajos (rojo): estados poco valiosos o peligrosos')
print('- El agente ha aprendido a evitar las trampas y a ir a la meta por caminos cortos')

## Probar el agente entrenado

In [ ]:
def test_agent(agent, env, n_episodes=10, verbose=True):
    """Evalúa el agente entrenado (sin exploración)"""
    test_rewards, test_lengths, success_count = [], [], 0
    original_epsilon = agent.epsilon
    agent.epsilon = 0.0  # solo explotación
    if verbose:
        print('Probando el agente entrenado (sin exploración):\n')
    for episode in range(n_episodes):
        state = env.reset()
        total_reward, steps, path = 0, 0, [state]
        for step in range(50):  # máximo 50 pasos
            action = agent.choose_action(state)
            next_state, reward, done = env.step(action)
            path.append(next_state)
            state = next_state
            total_reward += reward
            steps += 1
            if done:
                if next_state == env.goal_pos:
                    success_count += 1
                break
        test_rewards.append(total_reward)
        test_lengths.append(steps)
        if verbose:
            outcome = 'ÉXITO' if state == env.goal_pos else 'TRAMPA' if state in env.traps else 'TIEMPO'
            print(f"Episodio {episode + 1:2d}: {outcome:6s} | recompensa: {total_reward:6.1f} | pasos: {steps:2d} | camino: {' → '.join(str(p) for p in path)}")
    agent.epsilon = original_epsilon
    if verbose:
        print('\nResultados:')
        print(f'- Tasa de éxito: {success_count}/{n_episodes} ({success_count / n_episodes:.0%})')
        print(f'- Recompensa media: {np.mean(test_rewards):.2f} ± {np.std(test_rewards):.2f}')
        print(f'- Pasos medios: {np.mean(test_lengths):.1f} ± {np.std(test_lengths):.1f}')
    return test_rewards, test_lengths, success_count


test_rewards, test_lengths, success_count = test_agent(agent, env, n_episodes=10)

## Demostración: el agente entrenado recorre la rejilla

En lugar de dibujar una figura por paso, lo animamos. A la derecha, los cuatro valores Q del estado en el que está el robot.

In [ ]:
def demonstrate_episode(agent, env, max_steps=20):
    """Un episodio sin exploración, paso a paso, como animación"""
    original_epsilon, agent.epsilon = agent.epsilon, 0.0
    state, total, frames = env.reset(), 0, []
    frames.append(dict(pos=state, ret=0, a=None))
    print(f'Salida en {state}')
    for step in range(max_steps):
        q = agent.q_table[state]
        action = agent.choose_action(state)
        next_state, reward, done = env.step(action)
        total += reward
        print(f'Paso {step + 1}: estado {state} | Q = [' + ', '.join(f'{n}: {v:.2f}' for n, v in zip(ACTION_NAMES, q)) +
              f'] → {ACTION_NAMES[action]} | recompensa {reward} | acumulada {total}')
        frames.append(dict(pos=next_state, ret=total, a=action))
        state = next_state
        if done:
            print('  🎉 ¡ÉXITO! Ha llegado a la meta.' if state == env.goal_pos else '  💥 ¡FALLO! Ha caído en una trampa.')
            break
    agent.epsilon = original_epsilon

    fig, (axg, axb) = plt.subplots(1, 2, figsize=(8.5, 4), gridspec_kw=dict(width_ratios=[1.1, 1]))
    draw_grid(axg, env)
    robot = Circle(frames[0]['pos'][::-1], 0.25, color=COL['robot'], zorder=5); axg.add_patch(robot)
    trail, = axg.plot([], [], color=COL['path'], lw=3, alpha=0.4)
    bars = axb.bar(ACTION_NAMES, [0] * 4, color='#cbd5e1'); axb.set_ylim(-10, 10); axb.axhline(0, color=COL['muted'], lw=0.8)
    axb.set_title('Q(s, ·) en la posición actual')

    def upd(k):
        f = frames[k]
        p = np.array([fr['pos'] for fr in frames[:k + 1]])
        trail.set_data(p[:, 1], p[:, 0]); robot.center = (f['pos'][1], f['pos'][0])
        axg.set_title(f'Paso {k} · recompensa acumulada {f["ret"]:+d}')
        q = agent.q_table[f['pos']]
        for a, b in enumerate(bars):
            b.set_height(q[a]); b.set_color(COL['robot'] if a == int(np.argmax(q)) else '#cbd5e1')
        return []

    anim = animation.FuncAnimation(fig, upd, frames=range(len(frames)), interval=600)
    anim.upd, anim.frame_list, anim.fig = upd, list(range(len(frames))), fig
    plt.close(fig)
    show_anim(anim, gif='n15_agente_entrenado.gif', fps=2)
    print(f'\nEpisodio terminado en {len(frames) - 1} pasos con recompensa total {total}')
    return total, len(frames) - 1


print('=' * 60)
print('DEMOSTRACIÓN: el agente entrenado recorre la rejilla')
print('=' * 60)
demonstrate_episode(agent, env)

## 🎛️ Laboratorio: cambia los hiperparámetros y vuelve a entrenar

Entrenar 1000 episodios en esta rejilla tarda menos de un segundo, así que podemos experimentar. Mueve los *sliders*:
$\alpha$ (tasa de aprendizaje), $\gamma$ (descuento), el decaimiento de $\varepsilon$, la recompensa por paso y el número de episodios.
Si no tienes widgets, cambia los argumentos de `lab_qlearning(...)` y vuelve a ejecutar.

In [ ]:
def train_q(alpha=0.1, gamma=0.9, epsilon_decay=0.995, step_reward=-1.0, n_episodes=1000, seed=0, max_steps=50):
    """Entrena un agente nuevo y devuelve el agente, las curvas y el entorno."""
    random.seed(seed); np.random.seed(seed)
    e = GridWorld(step_reward=step_reward)
    ag = QLearningAgent(e.size, 4, learning_rate=alpha, discount_factor=gamma, epsilon=1.0)
    rewards, lengths, succ, goals = [], [], [], []
    for ep in range(n_episodes):
        s, tot, n = e.reset(), 0, 0
        for _ in range(max_steps):
            a = ag.choose_action(s); s2, r, d = e.step(a)
            ag.update_q_table(s, a, r, s2, d); s, tot, n = s2, tot + r, n + 1
            if d:
                break
        ag.decay_epsilon(decay_rate=epsilon_decay)
        rewards.append(tot); lengths.append(n); goals.append(s == e.goal_pos)
        if ep >= 99:
            succ.append(np.mean(goals[-100:]))
    return ag, rewards, lengths, succ, e


def greedy_success(ag, e, max_steps=50):
    """¿Llega el agente a la meta sin explorar? Devuelve (éxito, pasos)."""
    s = e.reset()
    for k in range(max_steps):
        s, _, d = e.step(int(np.argmax(ag.q_table[s])))
        if d:
            return s == e.goal_pos, k + 1
    return False, max_steps


def lab_qlearning(alpha=0.1, gamma=0.9, epsilon_decay=0.995, step_reward=-1.0, n_episodes=1000):
    ag, rw, ln, sc, e = train_q(alpha, gamma, epsilon_decay, step_reward, int(n_episodes))
    ok, steps = greedy_success(ag, e)
    first = next((i + 100 for i, v in enumerate(sc) if v >= 0.9), None)
    print(f'α={alpha} · γ={gamma} · decaimiento ε={epsilon_decay} · paso={step_reward} · {int(n_episodes)} episodios')
    print(f'Política final: {"✅ llega a la meta en %d pasos" % steps if ok else "❌ no llega a la meta"} | '
          f'éxito (últimos 100 ep.) {sc[-1] if sc else float("nan"):.0%} | 90 % de éxito por primera vez en el episodio {first}')
    plot_training(rw, ln, sc, ag.epsilon_history, ag.q_table, 'Laboratorio de Q-learning')


if USE_WIDGETS:
    sl = lambda v, a, b, s, d: widgets.FloatSlider(value=v, min=a, max=b, step=s, description=d, continuous_update=False,
                                                   style={'description_width': '120px'}, layout=widgets.Layout(width='360px'))
    w = dict(alpha=sl(0.1, 0.01, 1.0, 0.01, 'α (aprendizaje)'), gamma=sl(0.9, 0.0, 0.99, 0.01, 'γ (descuento)'),
             epsilon_decay=sl(0.995, 0.9, 0.9999, 0.0005, 'decaimiento ε'), step_reward=sl(-1.0, -5.0, 1.0, 0.25, 'recompensa/paso'),
             n_episodes=widgets.IntSlider(value=1000, min=50, max=3000, step=50, description='episodios', continuous_update=False,
                                          style={'description_width': '120px'}, layout=widgets.Layout(width='360px')))
    display(widgets.HBox([widgets.VBox([w['alpha'], w['gamma'], w['epsilon_decay']]), widgets.VBox([w['step_reward'], w['n_episodes']])]),
            widgets.interactive_output(lab_qlearning, w))
else:
    print('ipywidgets no disponible (o HEADLESS=1): versión estática. Cambia los argumentos y vuelve a ejecutar.')
    lab_qlearning(alpha=0.5, gamma=0.9)

In [ ]:
# Barrido rápido (sin widgets): ¿cuántos episodios tarda en alcanzar un 90 % de éxito según α y γ?
alphas, gammas = [0.05, 0.1, 0.3, 0.6, 1.0], [0.5, 0.7, 0.9, 0.99]
table = np.full((len(gammas), len(alphas)), np.nan)
for gi, g in enumerate(gammas):
    for ai, a in enumerate(alphas):
        _, _, _, sc, _ = train_q(a, g, n_episodes=1000)
        hit = next((i + 100 for i, v in enumerate(sc) if v >= 0.9), np.nan)
        table[gi, ai] = hit
fig, ax = plt.subplots(figsize=(6, 3.2))
im = ax.imshow(table, cmap='viridis_r'); ax.grid(False)
ax.set_xticks(range(len(alphas)), [str(a) for a in alphas]); ax.set_yticks(range(len(gammas)), [str(g) for g in gammas])
ax.set_xlabel('α'); ax.set_ylabel('γ'); ax.set_title('Episodio en que se alcanza un 90 % de éxito (gris: nunca)')
for gi in range(len(gammas)):
    for ai in range(len(alphas)):
        ax.text(ai, gi, '—' if np.isnan(table[gi, ai]) else f'{table[gi, ai]:.0f}', ha='center', va='center', color='white', fontsize=8)
plt.colorbar(im, ax=ax); plt.tight_layout(); plt.show()

### ✍️ Ejercicios
1. **Tasa de aprendizaje.** Con el laboratorio, compara α = 0.05, 0.1 y 0.6. ¿Cuál aprende antes? ¿Alguno da una curva más ruidosa? Explica por qué con la regla de actualización.
2. **Descuento.** Pon γ = 0. ¿Qué aprende el robot? ¿Y con γ = 0.99? Relaciónalo con lo «lejos» que mira el agente.
3. **Recompensa por paso.** Pon la recompensa por paso a 0 y luego a +1. ¿Sigue yendo a la meta? ¿Por qué? Es un ejemplo de lo importante (y peligroso) que es diseñar la recompensa.
4. **Exploración.** Con un decaimiento de ε de 0.9 (exploración que se apaga muy deprisa), ¿encuentra siempre el camino bueno? Prueba varias veces cambiando `seed` en `train_q`.

## Resumen e ideas clave

### Qué hemos aprendido

1. **Fundamentos del Q-learning**: el agente aprende a estimar la «calidad» (valor Q) de cada acción en cada estado.
2. **Exploración frente a explotación**: la estrategia ε-greedy equilibra probar acciones nuevas con aprovechar las que ya funcionan.
3. **Diseño de la recompensa**: la combinación de premio en la meta (+10), castigo en las trampas (−10) y coste por paso (−1) guía al agente hacia caminos eficientes.
4. **Convergencia**: con el tiempo la política se estabiliza y el agente llega siempre a la meta evitando las trampas.

### Componentes clave del Q-learning

- **Tabla Q**: guarda el valor aprendido de cada par estado-acción
- **Regla de actualización**: $Q(s,a) \leftarrow Q(s,a) + \alpha[r + \gamma \max_{a'} Q(s',a') - Q(s,a)]$
- **Estrategia de exploración**: política ε-greedy
- **Tasa de aprendizaje (α)**: lo rápido que el agente cambia lo que sabe
- **Factor de descuento (γ)**: lo que importan las recompensas futuras

### Aplicaciones reales

Este problema es muy simple, pero los mismos principios se aplican a:

- **Navegación de robots**: planificación de rutas en almacenes, vehículos autónomos
- **Juegos**: juegos de tablero, videojuegos
- **Gestión de recursos**: control de inventario, gestión de energía
- **Control de procesos**: automatización industrial, procesos químicos

### Siguientes pasos

Para ampliar este tutorial puedes:

1. **Modificar el entorno**: obstáculos móviles, varias metas u otras recompensas
2. **Probar otros algoritmos**: SARSA, Double Q-learning o Deep Q-Networks (DQN, n16)
3. **Experimentar con los parámetros**: otras tasas de aprendizaje, descuentos o estrategias de exploración

👉 Cuestionario: `n15_grid_robot_qlearning_tutorial_training_process_quiz.md`.